# Practical 5.1 — Wyvern Dragonette-004 Hyperspectral Satellite Imagery
**PLNT 7620 / SOIL 7220 · Satellite Remote Sensing Practicals · Digital AgroEcosystems Lab · University of Manitoba · Fall 2026**

**Source data:** [Shared Google Drive hyperspectral folder](https://drive.google.com/drive/folders/1RaPPPhmaJ8kOxwfLkoORBaR90-QBnVJJ?usp=sharing)  
**Sensor:** Dragonette-004 · **Product:** Level-2A surface reflectance · **Approximate size:** 1.82 GB GeoTIFF

**Duration:** ~120 minutes. Run the notebook **in order**. Each code section has an associated scientific interpretation task.

### Learning outcomes
1. Inspect a 31-band visible–near-infrared satellite dataset and read its actual band wavelengths.
2. Draw a **student-defined polygon** and subset the hyperspectral raster without reading the entire cube into RAM.
3. Compare true colour, colour infrared and red-edge composites.
4. Calculate NDVI, NDRE and chlorophyll-sensitive reflectance indices.
5. Compare a **single-pixel spectral curve** with **mean ± SD spectra from student-drawn sample polygons**.
6. Explain limitations of spatial resolution, band placement, reflectance quality and surface interpretation.

**Important:** Dragonette-004 provides VNIR imagery, **not SWIR**. Avoid claiming SWIR-derived moisture/mineral measurements. The formulas used below are educational vegetation proxies, not calibrated plant nutrient or chlorophyll predictions.

## 0 — Environment and dependencies
We will use Google Drive and Rasterio for data access, ipyleaflet for interactive drawing, and Matplotlib for publication-ready plots. Some browsers require third-party cookies / widget support for Colab interactive maps.

In [ ]:
%pip -q install rasterio ipyleaflet ipywidgets pyproj shapely matplotlib pandas
from google.colab import output
output.enable_custom_widget_manager()

import os, json, re, shutil, warnings, base64, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds, Window, transform as window_transform
from rasterio.features import geometry_mask
from rasterio.warp import transform_bounds, reproject, transform as raster_transform
from rasterio.crs import CRS
from shapely.geometry import shape, mapping, Point
from shapely.ops import transform as geom_transform
from pyproj import Transformer
from ipyleaflet import Map, ImageOverlay, DrawControl, LayersControl, basemaps, GeoJSON, Marker

OUTPUT_DIR = Path('/content/dragonette_5_1_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

## 1 — Mount Drive and locate your files
**Before running:** Open the shared folder above. Use **Organize → Add shortcut → My Drive** so the large image is accessible through your Colab Drive mount. A shared URL alone is **not** a mounted filesystem path. If you do not have access, ask your instructor to grant it.

You may paste the full path to the GeoTIFF in `IMAGE_PATH` below; otherwise this cell searches My Drive for files ending in `_l2a.tiff`. **Do not load the 1.82 GB file with `numpy.load` or `rasterio.read()` without a window.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Optional: set explicitly if your shortcut is elsewhere:
IMAGE_PATH = ''  # e.g. '/content/drive/MyDrive/Data/Hyperspectral/wyvern_..._l2a.tiff'
SEARCH_ROOT = Path('/content/drive/MyDrive')
if IMAGE_PATH:
    tif_path = Path(IMAGE_PATH)
else:
    matches = []
    for root, dirs, files in os.walk(SEARCH_ROOT):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for filename in files:
            if filename.lower().endswith('_l2a.tiff') and 'wyvern' in filename.lower():
                matches.append(Path(root) / filename)
        if len(matches) >= 20:
            break
    print('Candidate L2A files:')
    for i, candidate in enumerate(matches):
        print(i, candidate)
    if not matches:
        raise FileNotFoundError('No Wyvern *_l2a.tiff found. Add a My Drive shortcut or set IMAGE_PATH explicitly.')
    # If multiple products exist, set IMAGE_PATH explicitly and rerun.
    if len(matches) > 1:
        raise ValueError('Multiple scenes found; set IMAGE_PATH to the correct Dragonette-004 scene.')
    tif_path = matches[0]

if not tif_path.exists():
    raise FileNotFoundError(f'GeoTIFF not found: {tif_path}')
metadata_path = tif_path.with_suffix('.json')
data_mask_path = tif_path.with_name(tif_path.stem + '_data_mask.tiff')
quality_mask_path = tif_path.with_name(tif_path.stem + '_pixel_quality_mask.tiff')
print('Image:', tif_path)
print('JSON present:', metadata_path.exists())
print('Data mask present:', data_mask_path.exists())
print('Pixel quality mask present:', quality_mask_path.exists())

### Optional speed-up
Reading a cloud-optimized GeoTIFF through Google Drive FUSE can be slow. The notebook reads **reduced-resolution previews and windows**, not the entire image. If reads are very slow and your Colab disk has sufficient free space, change `COPY_LOCALLY` to `True` to copy the 1.82 GB GeoTIFF to the VM first. Copying is **not required**.

In [ ]:
COPY_LOCALLY = False
if COPY_LOCALLY:
    local_path = Path('/content') / tif_path.name
    free_bytes = shutil.disk_usage('/content').free
    if free_bytes < tif_path.stat().st_size * 1.5:
        raise RuntimeError('Insufficient Colab disk space to copy image.')
    if not local_path.exists() or local_path.stat().st_size != tif_path.stat().st_size:
        shutil.copy2(tif_path, local_path)
    tif_path = local_path
print('Working image:', tif_path)

## 2 — Inspect image structure, wavelengths, and surface reflectance
**Never assume band 1 means blue or band 31 means NIR.** Retrieve central wavelengths from the actual product metadata. This notebook requires valid wavelength information rather than silently inventing approximate values.

For Wyvern L2A, scaled integers may use a 0.0001 reflectance conversion and 65535 NoData. Check the file's own tags. The following cell prints metadata for inspection before you proceed.

In [ ]:
with rasterio.open(tif_path) as src:
    print('Bands:', src.count, 'Size:', src.width, 'x', src.height)
    print('CRS:', src.crs, 'Bounds:', src.bounds, 'Resolution:', src.res)
    print('dtype:', src.dtypes[0], 'NoData:', src.nodata)
    print('Tags:', src.tags())
    print('First 5 band descriptions:', src.descriptions[:5])
    print('First-band tags:', src.tags(1))
    print('Internal masks:', src.mask_flag_enums[:2])
    BANDS = src.count
    RASTER_CRS = src.crs
    RASTER_BOUNDS = src.bounds

if metadata_path.exists():
    with metadata_path.open('r') as f:
        metadata = json.load(f)
    print('JSON top-level keys:', list(metadata.keys()))
    print('JSON excerpt:', json.dumps(metadata, indent=2)[:2600])
else:
    metadata = {}

In [ ]:
def wavelength_values(obj, count, path='root'):
    """Find metadata wavelength vectors or per-band wavelength records; preserve original order."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = k.lower()
            if isinstance(v, list) and len(v) == count and ('wavelength' in key or 'center' in key):
                try:
                    values = [float(x) for x in v]
                    out.append((path + '.' + k, values))
                except (ValueError, TypeError):
                    pass
            if isinstance(v, list) and len(v) == count and all(isinstance(x, dict) for x in v):
                for attr in ('center_wavelength_nm', 'center_wavelength', 'wavelength_nm', 'wavelength', 'central_wavelength'):
                    if all(attr in x for x in v):
                        try:
                            out.append((path + '.' + k + '.' + attr, [float(x[attr]) for x in v]))
                        except (TypeError, ValueError):
                            pass
            out += wavelength_values(v, count, path + '.' + str(k))
    elif isinstance(obj, list):
        for i, x in enumerate(obj[:100]):
            if isinstance(x, dict):
                out += wavelength_values(x, count, path + '[' + str(i) + ']')
    return out

with rasterio.open(tif_path) as src:
    band_text = [src.descriptions[i] or src.tags(i+1).get('wavelength', '') for i in range(src.count)]
    tag_text = src.tags()
candidates = wavelength_values(metadata, BANDS)
if all(band_text):
    try:
        extracted = [float(re.search(r'\\d+(?:\\.\\d+)?', str(x)).group(0)) for x in band_text]
        candidates.insert(0, ('GeoTIFF band labels/tags', extracted))
    except (AttributeError, ValueError):
        pass

# A small number of metadata sources use micrometres instead of nanometres.
valid = []
for origin, vals in candidates:
    arr = np.array(vals, dtype=float)
    if len(arr) != BANDS or not np.all(np.isfinite(arr)):
        continue
    if 0.3 < np.nanmedian(arr) < 2.5:  # micrometres
        arr *= 1000
    if np.all((arr > 350) & (arr < 1100)) and np.all(np.diff(arr) > 0):
        valid.append((origin, arr))
print('Valid wavelength candidates:', [x[0] for x in valid])
# If metadata cannot be parsed, paste the exact BANDS wavelengths (nm) from product documentation.
MANUAL_WAVELENGTHS_NM = []  # not an invented evenly spaced wavelength series
if MANUAL_WAVELENGTHS_NM:
    WAVELENGTHS_NM = np.asarray(MANUAL_WAVELENGTHS_NM, dtype=float)
elif valid:
    WAVELENGTHS_NM = valid[0][1]
else:
    raise ValueError('Cannot reliably find band wavelengths. Inspect the JSON/GeoTIFF and supply MANUAL_WAVELENGTHS_NM.')
if len(WAVELENGTHS_NM) != BANDS or not np.all(np.diff(WAVELENGTHS_NM) > 0):
    raise ValueError('Wavelength count/order does not match image bands.')
print(pd.DataFrame({'Band (1-based)': range(1, BANDS+1), 'Centre wavelength (nm)': WAVELENGTHS_NM}).to_string(index=False))

In [ ]:
with rasterio.open(tif_path) as src:
    # Use the supplied documented L2A default unless your product explicitly differs.
    print('Review actual scale and offset tags:', src.scales[:4], src.offsets[:4])
    print('Review NoData:', src.nodata)
REFLECTANCE_SCALE = 0.0001  # default documented for Wyvern L2A; confirm above.
OVERRIDE_SCALE = None       # e.g. 1.0 if your scene truly stores unit reflectance.
if OVERRIDE_SCALE is not None:
    REFLECTANCE_SCALE = float(OVERRIDE_SCALE)
NODATA_FALLBACK = 65535
def reflectance(band_data, nodata=None):
    """Convert masked/scaled stored values to floating reflectance, invalid to NaN."""
    x = np.ma.asarray(band_data)
    data = np.asarray(x.data, dtype=np.float32)
    invalid = np.ma.getmaskarray(x) | ~np.isfinite(data)
    if nodata is not None:
        invalid |= data == nodata
    else:
        invalid |= data == NODATA_FALLBACK
    out = data * REFLECTANCE_SCALE
    out[invalid] = np.nan
    return out

def nearest_band(wavelength_nm):
    j = int(np.argmin(np.abs(WAVELENGTHS_NM - wavelength_nm)))
    return j + 1  # Rasterio bands are 1-indexed

for nm in (480, 550, 660, 700, 712, 735, 750, 800):
    j = nearest_band(nm)
    print(f'Target {nm} nm → band {j}, actual {WAVELENGTHS_NM[j-1]:.1f} nm')

## 3 — Display whole-scene composites (reduced-resolution)
The input can be large; we sample a maximum of about 800 pixels along the longest dimension for quick previews. **Visual stretching changes appearance, not underlying reflectance.**

We use a simple per-band percentile display stretch, masking invalid values first. The map overlay is reprojected to geographic coordinates so that student-drawn polygons align correctly with the raster.

In [ ]:
def rgb_from_cube(cube, wavelengths=(660, 550, 480), contrast=(2, 98)):
    channels = []
    for nm in wavelengths:
        idx = nearest_band(nm) - 1
        x = cube[idx]
        finite = x[np.isfinite(x)]
        if finite.size == 0:
            channels.append(np.zeros(x.shape, dtype=np.float32))
            continue
        lo, hi = np.percentile(finite, contrast)
        channels.append(np.clip((x-lo) / max(hi-lo, 1e-8), 0, 1))
    rgb = np.stack(channels, axis=-1)
    return np.nan_to_num(rgb, nan=0)

with rasterio.open(tif_path) as src:
    factor = min(1.0, 800 / max(src.width, src.height))
    p_width = max(1, int(src.width * factor))
    p_height = max(1, int(src.height * factor))
    preview_raw = src.read(out_shape=(src.count, p_height, p_width),
                           resampling=Resampling.average, masked=True)
    preview_cube = reflectance(preview_raw, src.nodata)
    preview_transform = src.transform * src.transform.scale(src.width / p_width, src.height / p_height)
    preview_crs = src.crs
    geobounds = transform_bounds(src.crs, 'EPSG:4326', *src.bounds, densify_pts=21)
del preview_raw

composites = {
    'Natural colour: 660/550/480': (660, 550, 480),
    'Colour infrared: 800/660/550': (800, 660, 550),
    'Red-edge: 800/735/660': (800, 735, 660)
}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, wavelengths) in zip(axes, composites.items()):
    ax.imshow(rgb_from_cube(preview_cube, wavelengths))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'scene_composites.png', dpi=180)
plt.show()

### Think and discuss
- Which composite most clearly distinguishes photosynthetically active vegetation from bare soil?
- Why is a red-edge composite scientifically different from a standard colour-infrared composite?
- What information is **lost** when all 31 bands are reduced to three display channels?

## 4 — Draw **your own study polygon**
Run the cell, zoom to your area, and draw **one polygon** using the polygon tool on the map. The satellite preview is overlaid on a street-map background. You can redraw; the **last polygon you create** will be used.

The polygon coordinates will be in WGS84 and converted automatically to the image CRS. If widgets do not work, the alternative is to export/draw a GeoJSON polygon externally and use the optional file-upload fallback below.

In [ ]:
def preview_overlay_data_url(cube, src_transform, src_crs, bounds_lonlat, out_max=650):
    """Reproject only 3 downsampled bands onto a north-up lat/lon display grid."""
    west, south, east, north = bounds_lonlat
    w = out_max
    h = max(100, round(w * (north-south) / max(east-west, 1e-8)))
    h = min(h, 850)
    from rasterio.transform import from_bounds as affine_bounds
    dst_transform = affine_bounds(west, south, east, north, w, h)
    source_rgb = rgb_from_cube(cube).astype('float32')
    out = np.zeros((h, w, 3), dtype='float32')
    for channel in range(3):
        reproject(source=source_rgb[:, :, channel], destination=out[:, :, channel],
                  src_transform=src_transform, src_crs=src_crs,
                  dst_transform=dst_transform, dst_crs='EPSG:4326',
                  src_nodata=None, dst_nodata=0,
                  resampling=Resampling.bilinear)
    buf = io.BytesIO()
    plt.imsave(buf, np.clip(out, 0, 1), format='png')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

west, south, east, north = geobounds
preview_url = preview_overlay_data_url(preview_cube, preview_transform, preview_crs, geobounds)
roi_geojson = None
map_roi = Map(center=((south+north)/2, (west+east)/2), zoom=13,
              basemap=basemaps.OpenStreetMap.Mapnik, scroll_wheel_zoom=True)
map_roi.layout.height = '550px'
map_roi.add(ImageOverlay(url=preview_url, bounds=((south, west), (north, east)), opacity=0.82))
draw_roi = DrawControl(polygon={'shapeOptions': {'color': '#ffcf33', 'weight': 3}},
                       rectangle={}, circle={}, circlemarker={}, polyline={}, marker={})
def on_roi_draw(target, action, geo_json):
    global roi_geojson
    if action == 'created' and geo_json['geometry']['type'] == 'Polygon':
        roi_geojson = geo_json['geometry']
        print('ROI polygon saved —', len(roi_geojson['coordinates'][0]), 'vertices')
    elif action == 'deleted':
        roi_geojson = None
        print('Polygon deleted. Draw a new one.')
draw_roi.on_draw(on_roi_draw)
map_roi.add(draw_roi)
display(map_roi)

In [ ]:
# Optional fallback, only if the interactive polygon did not register:
# from google.colab import files
# uploaded = files.upload()  # upload GeoJSON exported from a GIS or geojson.io
# geojson_obj = json.loads(next(iter(uploaded.values())).decode('utf-8'))
# feature = geojson_obj['features'][0] if geojson_obj['type'] == 'FeatureCollection' else geojson_obj
# roi_geojson = feature['geometry'] if feature['type'] == 'Feature' else feature
print('Polygon available:', roi_geojson is not None)

## 5 — Clip the selected polygon using an efficient windowed read
We transform the polygon into the raster CRS, read the smallest enclosing raster **window**, and mask pixels outside the polygon. If the selected region is too large for your Colab RAM, redraw a smaller polygon. We then save a compact, georeferenced 31-band subset.

This example uses the raster's own NoData mask. **Do not interpret the separate quality-mask codes without their product-specific definitions.**

In [ ]:
if roi_geojson is None:
    raise RuntimeError('Draw a polygon in Section 4 before proceeding.')
roi_shape_wgs = shape(roi_geojson)
if not roi_shape_wgs.is_valid or roi_shape_wgs.is_empty:
    raise ValueError('Draw a valid, non-self-intersecting polygon.')
with rasterio.open(tif_path) as src:
    to_native = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True).transform
    roi_native = geom_transform(to_native, roi_shape_wgs)
    left, bottom, right, top = roi_native.bounds
    # Window must intersect the scene; do not allow misleading empty reads.
    if not roi_native.intersects(shape({'type':'Polygon','coordinates':[[
        (src.bounds.left,src.bounds.bottom),(src.bounds.right,src.bounds.bottom),
        (src.bounds.right,src.bounds.top),(src.bounds.left,src.bounds.top),
        (src.bounds.left,src.bounds.bottom)]]})):
        raise ValueError('Selected polygon does not intersect the satellite scene.')
    window = from_bounds(left, bottom, right, top, src.transform)
    window = window.round_offsets().round_lengths()
    window = window.intersection(Window(0, 0, src.width, src.height))
    estimated_gib = src.count * window.width * window.height * 4 / (1024**3)
    print('Subset window:', int(window.width), 'x', int(window.height),
          '| float32 cube ~', round(estimated_gib, 2), 'GiB')
    if estimated_gib > 1.5:
        raise MemoryError('Polygon too large for a teaching subset. Draw a smaller polygon (< ~1.5 GiB).')
    subset_affine = window_transform(window, src.transform)
    subset_raw = src.read(window=window, masked=True)
    subset_cube = reflectance(subset_raw, src.nodata)
    subset_profile = src.profile.copy()
    subset_crs = src.crs
del subset_raw

inside = geometry_mask([mapping(roi_native)], out_shape=subset_cube.shape[1:],
                       transform=subset_affine, invert=True)
subset_cube[:, ~inside] = np.nan
valid_count = int((np.isfinite(subset_cube).all(axis=0)).sum())
if valid_count == 0:
    raise ValueError('No valid hyperspectral pixels within polygon. Draw another polygon.')
print('Valid all-band pixels:', valid_count)
subset_path = OUTPUT_DIR / 'Dragonette004_student_polygon_reflectance.tif'
subset_profile.update(driver='GTiff', dtype='float32', count=BANDS,
                      height=subset_cube.shape[1], width=subset_cube.shape[2],
                      transform=subset_affine, nodata=-9999, compress='deflate',
                      tiled=True, BIGTIFF='IF_SAFER')
with rasterio.open(subset_path, 'w', **subset_profile) as dst:
    dst.write(np.where(np.isfinite(subset_cube), subset_cube, -9999).astype('float32'))
    dst.update_tags(reflectance_units='unitless', description='Scaled L2A surface reflectance; masked outside polygon')
    for k, wl in enumerate(WAVELENGTHS_NM):
        dst.set_band_description(k+1, f'{wl:.2f} nm')
print('Saved:', subset_path)

## 6 — Visualize **your own clipped area**
Compare the three composites. Band central wavelengths are selected by actual scene metadata, not fixed band numbers. All panels show the **same geographic subset**.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, triplet) in zip(axs, composites.items()):
    rgb = rgb_from_cube(subset_cube, triplet)
    ax.imshow(rgb, interpolation='nearest')
    ax.set_title(name, fontsize=10)
    ax.set_axis_off()
plt.suptitle('Dragonette-004: Student-defined polygon subset')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'polygon_composites.png', dpi=200, bbox_inches='tight')
plt.show()

## 7 — Compute and compare vegetation indices
Use the **nearest actual available band** to the nominal target wavelength. Print actual wavelengths: ratios based on slightly different bands are not necessarily identical to canonical published indices.

- NDVI = (NIR − Red) / (NIR + Red)
- NDRE = (NIR − RedEdge) / (NIR + RedEdge)
- CIgreen = NIR / Green − 1
- CIred-edge = NIR / RedEdge − 1
- Red-edge slope = (R750 − R700) / (λ750 − λ700), in reflectance per nm

Index values can be affected by bare soil, phenology, mixed pixels, shadows, atmosphere, geometry and saturation. **Do not use a fixed red-edge formula as a validated chlorophyll estimate without calibration.**

In [ ]:
def R(nm):
    return subset_cube[nearest_band(nm)-1]
def safe_ratio(numerator, denominator):
    with np.errstate(divide='ignore', invalid='ignore'):
        x = np.divide(numerator, denominator,
                      out=np.full(numerator.shape, np.nan, dtype='float32'),
                      where=np.isfinite(denominator) & (np.abs(denominator) > 1e-9))
    return x

nir, red, green, rededge = R(800), R(660), R(550), R(712)
indices = {
    'NDVI': safe_ratio(nir-red, nir+red),
    'NDRE': safe_ratio(nir-rededge, nir+rededge),
    'CIgreen': safe_ratio(nir, green)-1,
    'CIred-edge': safe_ratio(nir, rededge)-1
}
wl700, wl750 = WAVELENGTHS_NM[nearest_band(700)-1], WAVELENGTHS_NM[nearest_band(750)-1]
indices['Red-edge slope'] = (R(750)-R(700)) / (wl750-wl700)
for name, arr in indices.items():
    arr[~inside] = np.nan
    print(name, '| median:', round(float(np.nanmedian(arr)), 4),
          '| valid:', int(np.isfinite(arr).sum()))

fig, axs = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, arr) in zip(axs.flat, indices.items()):
    if name in ('NDVI','NDRE'):
        lo, hi = -1, 1
    else:
        lo, hi = np.nanpercentile(arr, [2,98])
    im = ax.imshow(arr, cmap='viridis', vmin=lo, vmax=hi)
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=.045, pad=.02)
for ax in axs.flat[len(indices):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'hyperspectral_indices.png', dpi=180)
plt.show()

In [ ]:
for name, arr in indices.items():
    path = OUTPUT_DIR / (name.replace(' ','_').replace('-','_') + '.tif')
    idx_profile = subset_profile.copy()
    idx_profile.update(count=1, dtype='float32', nodata=-9999)
    with rasterio.open(path, 'w', **idx_profile) as ds:
        ds.write(np.where(np.isfinite(arr), arr, -9999).astype('float32'), 1)
print('Saved index GeoTIFFs to:', OUTPUT_DIR)

### Reflect on the results
1. Where do NDVI and NDRE agree, and where do they disagree?
2. Which additional portions of the VNIR spectrum would help discriminate vegetation types?
3. Why are red-edge indices **sensitive to** chlorophyll but not direct measurements of chlorophyll?

## 8 — Select individual pixels and draw smaller sample polygons
Use the map below to:
- **Click** an identifiable pixel within the subset. The last click is used for an **individual pixel signature**.
- Draw **2–3 separate sample polygons** over visually distinct areas (for example, vegetation and exposed soil, if present). These will become **polygon-mean reflectance signatures**.

Use the clipped image overlay as a guide. Clicks and polygons are converted from latitude/longitude to the raster coordinate system. Polygon means exclude invalid pixels and use the full 31-band spectrum.

In [ ]:
sample_polygons = []
sample_pixel_latlon = None
sample_map = Map(center=((south+north)/2, (west+east)/2), zoom=14,
                 basemap=basemaps.OpenStreetMap.Mapnik, scroll_wheel_zoom=True)
sample_map.layout.height = '550px'
sample_map.add(ImageOverlay(url=preview_url, bounds=((south, west), (north, east)), opacity=0.83))
draw_samples = DrawControl(polygon={'shapeOptions': {'color':'#00c7a7'}},
                           rectangle={}, polyline={}, circle={}, circlemarker={}, marker={})
def on_sample_draw(target, action, feature):
    global sample_polygons
    if action == 'created' and feature['geometry']['type']=='Polygon':
        sample_polygons.append(feature['geometry'])
        print(f'Sample polygon {len(sample_polygons)} added')
    if action == 'deleted':
        sample_polygons = []
        print('Sample polygons reset (redraw polygons for analysis)')
def on_sample_click(**kwargs):
    global sample_pixel_latlon
    if kwargs.get('type') == 'click':
        sample_pixel_latlon = kwargs.get('coordinates')
        print('Last sample pixel (lat,lon):', sample_pixel_latlon)
draw_samples.on_draw(on_sample_draw)
sample_map.on_interaction(on_sample_click)
sample_map.add(draw_samples)
display(sample_map)

## 9 — Plot **individual-pixel** and **polygon-mean ± SD** spectra
If any sample falls outside the selected ROI or contains too few valid pixels, the code reports it rather than fabricating a curve. Means and standard deviations are computed separately for each band across valid pixels inside each sample polygon.

In [ ]:
if sample_pixel_latlon is None:
    raise RuntimeError('Click one pixel on the map in Section 8.')
if len(sample_polygons) < 2:
    raise RuntimeError('Draw at least two sample polygons in Section 8.')

to_native_coords = Transformer.from_crs('EPSG:4326', subset_crs, always_xy=True).transform
lat, lon = sample_pixel_latlon
x, y = to_native_coords(lon, lat)
row, col = rasterio.transform.rowcol(subset_affine, x, y)
if not (0 <= row < subset_cube.shape[1] and 0 <= col < subset_cube.shape[2]):
    raise ValueError('Selected pixel is outside the subset. Click inside the polygon.')
pixel_spectrum = subset_cube[:, row, col]
if not np.isfinite(pixel_spectrum).all():
    raise ValueError('Clicked pixel is invalid or outside your first polygon. Choose a valid pixel.')

spectral_records = []
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(WAVELENGTHS_NM, pixel_spectrum, marker='o', markersize=3,
        color='black', linewidth=1.5, label='Individual pixel')
for k, poly in enumerate(sample_polygons, 1):
    poly_native = geom_transform(to_native_coords, shape(poly))
    sample_inside = geometry_mask([mapping(poly_native)],
                                  out_shape=subset_cube.shape[1:],
                                  transform=subset_affine, invert=True)
    spectra = subset_cube[:, sample_inside].T
    valid_rows = np.isfinite(spectra).all(axis=1)
    spectra = spectra[valid_rows]
    if spectra.shape[0] < 2:
        print(f'Polygon {k} skipped: fewer than 2 valid all-band pixels.')
        continue
    mean = spectra.mean(axis=0)
    sd = spectra.std(axis=0, ddof=1)
    (line,) = ax.plot(WAVELENGTHS_NM, mean, linewidth=2,
                       label=f'Polygon {k} mean (n={len(spectra)})')
    ax.fill_between(WAVELENGTHS_NM, mean-sd, mean+sd, color=line.get_color(), alpha=.14)
    for wl, mu, sigma in zip(WAVELENGTHS_NM, mean, sd):
        spectral_records.append({'sample':f'Polygon {k}', 'wavelength_nm':wl,
                                 'mean_reflectance':mu, 'sd_reflectance':sigma,
                                 'n_pixels':len(spectra)})
for wl, val in zip(WAVELENGTHS_NM, pixel_spectrum):
    spectral_records.append({'sample':'Individual pixel', 'wavelength_nm':wl,
                             'mean_reflectance':val, 'sd_reflectance':np.nan,
                             'n_pixels':1})
if not any(r['n_pixels'] > 1 for r in spectral_records):
    raise ValueError('No valid sample polygons. Draw polygons over actual valid image data.')
ax.set(xlabel='Centre wavelength (nm)', ylabel='L2A surface reflectance',
       title='Dragonette-004 — Individual pixel vs polygon mean ± SD')
ax.grid(alpha=.25)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'spectral_signatures.png', dpi=200)
plt.show()
spectral_df = pd.DataFrame(spectral_records)
spectral_df.to_csv(OUTPUT_DIR/'spectral_signatures.csv', index=False)
display(spectral_df.head(12))

## 10 — Save your ROI and share the results
The output directory contains clipped spectral data, spectral index GeoTIFFs, RGB figures and a CSV of spectral signatures. **Colab temporary files disappear when the session ends.** Copy outputs to your own Google Drive folder before ending the session.

In [ ]:
with (OUTPUT_DIR/'student_ROI.geojson').open('w') as f:
    json.dump({'type':'FeatureCollection','features':[{
        'type':'Feature','properties':{'name':'Student ROI'},'geometry':roi_geojson
    }]}, f, indent=2)
sample_fc = {'type':'FeatureCollection','features':[
    {'type':'Feature','properties':{'name':f'Sample {i+1}'},'geometry':poly}
    for i,poly in enumerate(sample_polygons)]}
with (OUTPUT_DIR/'student_sample_polygons.geojson').open('w') as f:
    json.dump(sample_fc, f, indent=2)
print('Saved outputs:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p.name, f'({p.stat().st_size / (1024**2):.1f} MB)')
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    destination = Path('/content/drive/MyDrive/Dragonette_Practical_5_1_Results')
    destination.mkdir(parents=True, exist_ok=True)
    for p in OUTPUT_DIR.iterdir():
        if p.is_file():
            shutil.copy2(p, destination/p.name)
    print('Copied to:', destination)

## Guided discussion and interpretation (submit short answers)
1. **Image interpretation:** Compare your natural-colour, colour-infrared and red-edge images. Which features become clearer, and why?
2. **Spatial sampling:** How many full-resolution pixels were inside your study polygon? What does ~5 m spatial resolution mean for mixed pixels?
3. **Index comparison:** Compare NDVI with NDRE for a distinctive part of your ROI. Explain a difference using red versus red-edge reflectance.
4. **Spectral curves:** Compare one individual pixel to two or more polygon means. Which spectral regions provide the strongest discrimination? How do the SD ribbons affect your confidence?
5. **Sensor limitations:** Identify at least **two limitations** of this L2A VNIR image for diagnosing crops or soil properties.
6. **Scientific caution:** Which of your conclusions would require independent field measurements before being described as a validated crop condition assessment?

### Deliverables
- One screenshot of your student-drawn ROI and the three composites.
- Index maps (at least NDVI, NDRE and one chlorophyll-sensitive index).
- One figure with a pixel spectrum and **at least two polygon-mean ± SD spectra**.
- Your short written interpretation (approximately 250–400 words).
- Optional: exported GeoTIFF subset and spectral CSV.

### Reproducibility note
Save a notebook copy with your own work. Record the original scene ID, actual wavelengths and location/extent of your study polygon. Dataset use and redistribution remain subject to the provider's licensing and instructor access terms.